# Analisi dati e machine learning - Qualita dell'aria

Questo notebook spiega il progetto in italiano. L'obiettivo e capire il dataset, leggere i grafici, interpretare le variabili e costruire un modello di regressione per stimare concentrazioni di gas a partire da risposte di sensori chimici.

Il dataset contiene misure orarie raccolte da un dispositivo multisensore installato in una zona urbana inquinata in Italia. Le colonne `GT` sono misure di riferimento fornite da un analizzatore certificato. Le colonne `PT08...` sono risposte dei sensori chimici.

## 1. Import e caricamento dati

Il codice sotto importa le funzioni gia presenti nel progetto. Questo e importante perche il notebook non duplica la logica dell'app: usa gli stessi moduli Python della dashboard.

In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.config import DEFAULT_TARGET, REFERENCE_COLUMNS, SENSOR_COLUMNS, SUPPORTED_TARGETS
from src.models.regression import evaluate_models
from src.utils.analysis import (
    build_dataset_profile,
    monthly_missingness,
    numeric_summary,
    sensor_reference_correlation,
    variable_type_summary,
)
from src.utils.data_loader import load_air_quality_data, summarize_data_quality
from src.utils.preprocessing import build_modeling_frame, missingness_table

df = load_air_quality_data()
df.head()

## 2. Cosa fa il programma

Il programma fa quattro cose principali:

1. carica il file originale `AirQualityUCI.csv`;
2. converte `Date` e `Time` in una colonna temporale chiamata `timestamp`;
3. converte i valori `-200` in valori mancanti reali;
4. crea un workflow di analisi e regressione per stimare concentrazioni di gas.

Questo non e un sistema ufficiale di monitoraggio ambientale. E un progetto di analisi dati e ML per mostrare come lavorare con dati scientifici rumorosi.

In [ ]:
profile = build_dataset_profile(df)
quality = summarize_data_quality()
profile, quality

## 3. Dizionario delle variabili

| Variabile | Significato pratico | Tipo |
|---|---|---|
| `timestamp` | Data e ora della misura | tempo |
| `CO(GT)` | concentrazione di monossido di carbonio misurata dall'analizzatore certificato | riferimento |
| `NMHC(GT)` | idrocarburi non metanici misurati dall'analizzatore certificato | riferimento |
| `C6H6(GT)` | benzene misurato dall'analizzatore certificato | riferimento |
| `NOx(GT)` | ossidi di azoto totali misurati dall'analizzatore certificato | riferimento |
| `NO2(GT)` | biossido di azoto misurato dall'analizzatore certificato | riferimento |
| `PT08.S1(CO)` | risposta del sensore chimico sensibile al CO | sensore |
| `PT08.S2(NMHC)` | risposta del sensore chimico sensibile agli NMHC | sensore |
| `PT08.S3(NOx)` | risposta del sensore chimico sensibile agli NOx | sensore |
| `PT08.S4(NO2)` | risposta del sensore chimico sensibile al NO2 | sensore |
| `PT08.S5(O3)` | risposta del sensore chimico sensibile all'ozono | sensore |
| `T` | temperatura | meteo |
| `RH` | umidita relativa | meteo |
| `AH` | umidita assoluta | meteo |

`GT` significa ground truth: sono le misure di riferimento. `PT08` indica risposte dei sensori, non concentrazioni certificate.

## 4. Variabili per tipo

Questo grafico serve a capire se il dataset contiene variabili numeriche, categoriche o temporali. Dopo la pulizia, quasi tutte le variabili sono numeriche perche il dataset e composto da misure di sensori, gas e meteo.

In [ ]:
type_summary = variable_type_summary(df)
type_summary

In [ ]:
px.pie(type_summary, names="variable_type", values="count", hole=0.45, title="Variabili per tipo")

**Interpretazione:** se vedi quasi solo variabili numeriche, e normale. Questo progetto non e come Titanic, dove ci sono molte categorie come sesso, classe o porto. Qui il problema e scientifico e quantitativo: misurare e stimare concentrazioni.

## 5. Valori mancanti

Nel dataset originale i valori mancanti sono codificati con `-200`. Il programma li trasforma in missing values reali. Questa scelta e importante: se lasciassimo `-200`, il modello penserebbe che sia una misura reale molto bassa.

In [ ]:
missing = missingness_table(df)
missing

In [ ]:
px.bar(
    missing[missing["missing_values"] > 0],
    x="column",
    y="missing_percent",
    title="Percentuale di valori mancanti per variabile",
)

**Interpretazione:** una variabile con tanti valori mancanti e meno affidabile per un modello. In questo dataset `NMHC(GT)` tende ad avere molti valori mancanti, quindi non e una buona scelta come target di default. Per un colloquio, questa e una buona osservazione da spiegare: non basta addestrare un modello, bisogna prima capire la qualita del dato.

## 6. Valori mancanti nel tempo

Il grafico seguente mostra se i missing values sono concentrati in certi mesi. Questo puo indicare problemi di acquisizione, manutenzione dello strumento o periodi meno affidabili.

In [ ]:
monthly = monthly_missingness(df)
px.line(monthly, x="month", y="missing_cells", markers=True, title="Valori mancanti per mese")

**Interpretazione:** se una zona del grafico e molto piu alta, significa che in quel periodo ci sono piu problemi nei dati. Per un dataset di sensori questo e importante perche la qualita del dato puo cambiare nel tempo.

## 7. Aria pulita o inquinata?

In questo progetto possiamo fare una lettura qualitativa, non una classificazione ufficiale.

- valori piu bassi di `CO(GT)`, `C6H6(GT)`, `NOx(GT)` e `NO2(GT)` indicano generalmente aria piu pulita per questi inquinanti;
- valori piu alti indicano episodi piu inquinati o maggiore influenza di traffico e combustione;
- `C6H6(GT)` e benzene, quindi e importante per la calibrazione discussa nel paper originale;
- `NOx(GT)` e `NO2(GT)` sono collegati, ma non sono la stessa cosa.

Non bisogna dire che l'app certifica se l'aria e pulita o pericolosa. Per una valutazione normativa servono medie su finestre temporali specifiche, strumenti validati e soglie ufficiali aggiornate.

## 8. Relazione sensore - riferimento

Ora confrontiamo una risposta sensore con una misura certificata. Se il sensore segue bene la misura certificata, allora puo essere utile per costruire un modello di calibrazione.

In [ ]:
sensor = "PT08.S2(NMHC)"
target = DEFAULT_TARGET
scatter_df = df[[sensor, target]].dropna()

fig = px.scatter(scatter_df, x=sensor, y=target, opacity=0.35, title=f"{sensor} vs {target}")
slope, intercept = np.polyfit(scatter_df[sensor], scatter_df[target], deg=1)
x_line = np.array([scatter_df[sensor].min(), scatter_df[sensor].max()])
fig.add_trace(go.Scatter(x=x_line, y=slope * x_line + intercept, mode="lines", name="Trend line"))
fig

**Interpretazione:** se i punti seguono una direzione chiara, il sensore contiene informazione utile. Se i punti sono molto dispersi, significa che c'e rumore, cross-sensitivity o influenza di variabili esterne come temperatura e umidita. La linea e solo una guida visiva, non il modello finale.

## 9. Correlazione Pearson tra sensori e gas

La matrice di correlazione Pearson mostra relazioni lineari tra variabili. Serve come diagnostica iniziale, non come prova definitiva.

In [ ]:
corr = sensor_reference_correlation(df, method="pearson")
px.imshow(corr, text_auto=".2f", aspect="auto", title="Matrice di correlazione Pearson: sensori e misure di riferimento")

**Interpretazione:** valori vicini a `1` indicano che due variabili tendono a crescere insieme; valori vicini a `-1` indicano relazione inversa; valori vicini a `0` indicano assenza di relazione lineare forte. Nei sensori chimici la correlazione puo essere influenzata da drift e cross-sensitivity.

## 10. Preparazione per il modello ML

Il modello usa risposte dei sensori, variabili meteo e feature temporali. Non usa altre colonne `GT` come input, per evitare una forma di data leakage: sarebbe troppo facile usare una misura certificata per predire un'altra misura certificata.

In [ ]:
X, y = build_modeling_frame(df, DEFAULT_TARGET)
X.head(), y.head(), X.shape, y.shape

In [ ]:
X.columns.tolist()

## 11. Benchmark dei modelli di regressione

Il progetto confronta modelli di regressione usando uno split cronologico: il modello impara dai dati piu vecchi e viene testato sui dati piu recenti. Questa scelta e piu realistica per dati temporali con possibile drift.

In [ ]:
run = evaluate_models(df, target=DEFAULT_TARGET, test_size=0.2, cv_splits=5)
run.leaderboard

**Interpretazione delle metriche:**

- `MAE`: errore medio assoluto, nella stessa unita del target;
- `RMSE`: penalizza di piu gli errori grandi;
- `R2`: misura quanta variabilita viene spiegata dal modello;
- `Dummy median`: baseline semplice; un modello utile dovrebbe batterla.

Questo e un benchmark serio per portfolio, ma non e ancora uno studio scientifico completo: mancano drift correction, incertezza predittiva e validazione piu approfondita.